In [1]:
!pip install llama-parse llama-index

In [2]:
!pip install nest_asyncio

In [3]:
import json
import os
from pathlib import Path

from llama_parse import LlamaParse
import nest_asyncio

nest_asyncio.apply()

/tmp/ipykernel_25255/3011005053.py:5: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


In [4]:
LLAMA_CLOUD_API_KEY = "llx-KhLCHZrD2Pdt1bncbZn98jC24K1NqLhyefnnRfArIl4NLasu"

DATA_FOLDER = Path("../data")

OUTPUT_FOLDER = Path("../parsed_output")

OUTPUT_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)

In [5]:
parser = LlamaParse(
    api_key=LLAMA_CLOUD_API_KEY,
    result_type="markdown",
    verbose=True
)

In [6]:
pdf_files = list(
    DATA_FOLDER.rglob("*.pdf")
)

print(f"Total PDFs Found: {len(pdf_files)}")

for pdf in pdf_files[:10]:
    print(pdf)

Total PDFs Found: 36
../data/Industry_All_risk/INDUSTRIAL ALL RISKS - Policy Wordings.pdf
../data/Industry_All_risk/Laghu - Policy wordings.pdf
../data/Industry_All_risk/pw_msme_pdf.pdf
../data/Industry_All_risk/industrial-all-risk.pdf
../data/Industry_All_risk/Industrial-All-Risk.pdf
../data/Industry_All_risk/Policy wordings.pdf
../data/Liability/public-liability-insurance-(under-public-liability-insurance-act-1991)_retail8c0007ff45fd68ff8a0df0055f8783a7.pdf
../data/Liability/74. Product Liability - Policy Wording_GEN403.pdf
../data/Liability/public-liability-industrial-storage-risks-prospectus.pdf
../data/Liability/7a3f930367ac4034993dc16cb124b261.pdf


In [7]:
for pdf in pdf_files[:10]:

    print(
        f"PDF: {pdf.name}"
    )

    print(
        f"Category: {pdf.parent.name}"
    )

    print("-" * 50)

PDF: INDUSTRIAL ALL RISKS - Policy Wordings.pdf
Category: Industry_All_risk
--------------------------------------------------
PDF: Laghu - Policy wordings.pdf
Category: Industry_All_risk
--------------------------------------------------
PDF: pw_msme_pdf.pdf
Category: Industry_All_risk
--------------------------------------------------
PDF: industrial-all-risk.pdf
Category: Industry_All_risk
--------------------------------------------------
PDF: Industrial-All-Risk.pdf
Category: Industry_All_risk
--------------------------------------------------
PDF: Policy wordings.pdf
Category: Industry_All_risk
--------------------------------------------------
PDF: public-liability-insurance-(under-public-liability-insurance-act-1991)_retail8c0007ff45fd68ff8a0df0055f8783a7.pdf
Category: Liability
--------------------------------------------------
PDF: 74. Product Liability - Policy Wording_GEN403.pdf
Category: Liability
--------------------------------------------------
PDF: public-liability-ind

In [8]:
def parse_policy(pdf_path):

    docs = parser.load_data(
        str(pdf_path)
    )
    if len(docs) == 0:

        raise ValueError(
            "No pages returned by LlamaParse"
        )

    pages = []

    full_text = []

    for page_num, doc in enumerate(
        docs,
        start=1
    ):

        pages.append(
            {
                "page_number": page_num,
                "content": doc.text
            }
        )

        full_text.append(
            doc.text
        )

    return {

        "source_file":
            pdf_path.name,

        "document_id":
            pdf_path.stem,

        "insurance_category":
            pdf_path.parent.name,

        "page_count":
            len(pages),

        "raw_text":
            "\n\n".join(full_text),

        "pages":
            pages
    }

In [9]:
sample_pdf = pdf_files[0]

policy = parse_policy(sample_pdf)

print("Source File:", policy["source_file"])
print("Category:", policy["insurance_category"])
print("Pages:", policy["page_count"])
print("Raw Text Length:", len(policy["raw_text"]))

Started parsing the file under job_id 413722ce-b378-43f9-9a3d-c779e7669ab6
Source File: INDUSTRIAL ALL RISKS - Policy Wordings.pdf
Category: Industry_All_risk
Pages: 18
Raw Text Length: 46251


In [ ]:
sample_output = (
    OUTPUT_FOLDER /
    "sample_output.json"
)

with open(
    sample_output,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        policy,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Saved")

In [ ]:
results = []

for pdf_file in pdf_files:

    try:

        print(
            f"\nProcessing: {pdf_file.name}"
        )

        policy_json = parse_policy(
            pdf_file
        )

        output_path = (
            OUTPUT_FOLDER /
            f"{pdf_file.stem}.json"
        )

        with open(
            output_path,
            "w",
            encoding="utf-8"
        ) as f:

            json.dump(
                policy_json,
                f,
                indent=2,
                ensure_ascii=False
            )

        results.append(
            {
                "file": pdf_file.name,
                "category": pdf_file.parent.name,
                "pages": policy_json["page_count"],
                "raw_text_length": len(
                    policy_json["raw_text"]
                ),
                "status": "SUCCESS"
            }
        )

        print("Saved")

    except Exception as e:

        results.append(
            {
                "file": pdf_file.name,
                "status": "FAILED",
                "error": str(e)
            }
        )

        print(
            f"FAILED: {e}"
        )

In [ ]:
!pip install pandas
import pandas as pd

summary_df = pd.DataFrame(
    results
)

summary_df

In [ ]:
failed = summary_df[
    summary_df["status"] == "FAILED"
]

failed

In [ ]:
pdf_path = next(
    p for p in pdf_files
    if p.name == "fire-special-perils.pdf"
)

docs = parser.load_data(
    str(pdf_path)
)

print(type(docs))
print(len(docs))

In [ ]:
pdf_path = next(
    p for p in pdf_files
    if p.name == "fire-special-perils.pdf"
)

docs = parser.load_data(str(pdf_path))

print(len(docs))

In [10]:
pdf_path = next(
    p for p in pdf_files
    if p.name == "Laghu - Policy wordings.pdf"
)

for i in range(3):

    print(f"\nRun {i+1}")

    parser = LlamaParse(
        api_key=LLAMA_CLOUD_API_KEY,
        result_type="markdown",
        verbose=True
    )

    docs = parser.load_data(str(pdf_path))

    print("Pages:", len(docs))


Run 1
Started parsing the file under job_id f2b353eb-19a9-4e8b-bc0c-3eb06a8f97e6
Pages: 26

Run 2
Started parsing the file under job_id f2b353eb-19a9-4e8b-bc0c-3eb06a8f97e6
Pages: 26

Run 3
Started parsing the file under job_id f2b353eb-19a9-4e8b-bc0c-3eb06a8f97e6
Pages: 26
